In [ ]:
# Colab reproducibility — uncomment to clone the repo and enter this folder:
# !git clone https://github.com/JuliettKhar/mlops-zoomcamp-exercises.git
# %cd mlops-zoomcamp-exercises/02-experiment-tracking

In [ ]:
# Colab setup — uncomment to install packages not preinstalled in Colab:
# !pip install mlflow

In [1]:
import mlflow

In [5]:
from mlflow.tracking import MlflowClient
MLFLOW_TACKING_URI = 'sqlite:///mlflow.db'

client = MlflowClient(tracking_uri=MLFLOW_TACKING_URI)

In [15]:
for exp in client.search_experiments():
    print(exp.name, exp.experiment_id)

my-new-experiment-3 4
my-new-experiment-2 3
my-new-experiment 2
Default 0


In [14]:
client.create_experiment(name='my-new-experiment-3')

'4'

In [16]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids=['2'],
    filter_string='',
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=['metrics.rmse ASC']
)

In [18]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 754c019a8c214cb3b8938b4236cea092, rmse: 63.4267
run id: 595a8a5fb99d4d48bebb43d84c254eec, rmse: 63.4745
run id: 099d111cf41e4e23932a78a8585e308f, rmse: 63.4858
run id: 77f7c8fa2c2142b1ad855216b5127cda, rmse: 63.5191
run id: b1b611c5552e448497c5d05bc98f8b23, rmse: 63.5663


In [26]:
run_id = "e20560ecbd23409cbb17d32899ae7435"
model_uri = f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri=model_uri, name="xgboost-taxi-regressor-2.1")

Successfully registered model 'xgboost-taxi-regressor-2.1'.
2026/05/26 10:00:58 WARNING mlflow.tracking._model_registry.fluent: Run with id e20560ecbd23409cbb17d32899ae7435 has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-aff75da99c4241f5ac20285c5fc14099 instead
Created version '1' of model 'xgboost-taxi-regressor-2.1'.


<ModelVersion: aliases=[], creation_timestamp=1779757258436, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1779757258436, metrics=None, model_id=None, name='xgboost-taxi-regressor-2.1', params=None, run_id='e20560ecbd23409cbb17d32899ae7435', run_link=None, source='models:/m-aff75da99c4241f5ac20285c5fc14099', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [27]:
model_uri

'runs:/e20560ecbd23409cbb17d32899ae7435/models_mlflow'

In [28]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1779756418083, deployment_job_id=None, deployment_job_state=None, description=None, last_updated_timestamp=1779756418083, latest_versions=[], name='my-new-experiment', tags={}, workspace='default'>,
 <RegisteredModel: aliases={}, creation_timestamp=1779751461101, deployment_job_id=None, deployment_job_state=None, description='the nyc taxi duration prediction', last_updated_timestamp=1779751828107, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1779751547369, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1779751547369, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='70839bde3fce412abe18aa869761f347', run_link='', source='models:/m-3cd6256d60494adba46a3cd5bc4f0bae', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>], name='nyc-taxi-regressor', tags={}, workspace='default'>,
 <RegisteredModel: aliases={}, creatio

In [32]:
model_name = 'nyc-taxi-regressor'
# latest_versions = client.get_latest_versions(name=model_name)
latest_versions =  client.search_model_versions(f"name='{model_name}'")

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: None
version: 2, stage: None


In [34]:
client.transition_model_version_stage(
    name=model_name,
    version='2',
    stage='Staging',
    archive_existing_versions=False
)

/var/folders/gk/87pgnywd0yj8v_zr8sbrmdd40000gn/T/ipykernel_32721/2640946248.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1779751547369, current_stage='Staging', deployment_job_state=None, description='', last_updated_timestamp=1779759051812, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='70839bde3fce412abe18aa869761f347', run_link='', source='models:/m-3cd6256d60494adba46a3cd5bc4f0bae', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [35]:
client.update_model_version(
    name=model_name,
    version='2',
    description="The model version was transitioned to new stage"
)

<ModelVersion: aliases=[], creation_timestamp=1779751547369, current_stage='Staging', deployment_job_state=None, description='The model version was transitioned to new stage', last_updated_timestamp=1779759660536, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='70839bde3fce412abe18aa869761f347', run_link='', source='models:/m-3cd6256d60494adba46a3cd5bc4f0bae', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [37]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version='2',
    description=f"The model version was transitioned to new stage on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1779751547369, current_stage='Staging', deployment_job_state=None, description='The model version was transitioned to new stage on 2026-05-26', last_updated_timestamp=1779759852428, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='70839bde3fce412abe18aa869761f347', run_link='', source='models:/m-3cd6256d60494adba46a3cd5bc4f0bae', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [38]:
import pandas as pd
import pickle
from sklearn.metrics import root_mean_squared_error

In [39]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df["duration"] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    return df

In [50]:
def preprocess(df, dv):
    df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]
    categorical = ['PU_DO']
    numerical = ['trip_distance']

    dicts = df[categorical + numerical].to_dict(orient="records")

    return dv.transform(dicts)

In [51]:
def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)

    return {"rmse": root_mean_squared_error(y_test, y_pred)}


In [52]:
df = read_dataframe("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet")

In [53]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration
0,2,2021-03-01 00:05:42,2021-03-01 00:14:03,N,1.0,83,129,1.0,1.56,7.5,...,0.5,0.00,0.0,None,0.3,8.80,1.0,1.0,0.0,8.350000
1,2,2021-03-01 00:21:03,2021-03-01 00:26:17,N,1.0,243,235,1.0,0.96,6.0,...,0.5,0.00,0.0,None,0.3,7.30,2.0,1.0,0.0,5.233333
2,2,2021-03-01 00:02:06,2021-03-01 00:22:26,N,1.0,75,242,1.0,9.93,28.0,...,0.5,2.00,0.0,None,0.3,31.30,1.0,1.0,0.0,20.333333
3,2,2021-03-01 00:24:03,2021-03-01 00:31:43,N,1.0,242,208,1.0,2.57,9.5,...,0.5,0.00,0.0,None,0.3,10.80,2.0,1.0,0.0,7.666667
4,1,2021-03-01 00:11:10,2021-03-01 00:14:46,N,1.0,41,151,1.0,0.80,5.0,...,0.5,1.85,0.0,None,0.3,8.15,1.0,1.0,0.0,3.600000


In [54]:
run_id = '11b501bb9d3845389d1d65bf73a03e29'
client.download_artifacts(
    run_id=run_id,
    path='preprocessor',
    dst_path='.'
)

'/Users/julia/Desktop/Projects/mlops-zoomcamp-exercises/02-experiment-tracking/preprocessor'

In [55]:
with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)


In [56]:
X_test = preprocess(df, dv)
y_test = df['duration'].values

In [57]:
test_model(
    name="nyc-taxi-regressor",
    stage="Staging",
    X_test=X_test,
    y_test=y_test
)

{'rmse': 30.061080315418092}